# 코사인 유사도를 이용한 추천 시스템

## Numpy를 이용해서 코사인 유사도를 계산하는 함수를 구현

### 예시 문서
문서 1 : 저는 사과 좋아요  
문서 2 : 저는 바나나 좋아요  
문서 3 : 저는 바나나 좋아요 저는 바나나 좋아요  
  
띄어쓰기 기준 토큰화  
||바나나|사과|저는|좋아요|
|--|--|--|--|--|
|문서 1|0|1|1|1|
|문서 2|1|0|1|1|
|문서 3|2|0|2|2|

In [1]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

doc1 = np.array([0, 1, 1, 1])
doc2 = np.array([1, 0, 1, 1])
doc3 = np.array([2, 0, 2, 2])

print('문서 1과 문서 2의 코사인 유사도:', cos_sim(doc1, doc2))
print('문서 1과 문서 3의 코사인 유사도:', cos_sim(doc1, doc3))
print('문서 2과 문서 3의 코사인 유사도:', cos_sim(doc2, doc3))

문서 1과 문서 2의 코사인 유사도: 0.6666666666666667
문서 1과 문서 3의 코사인 유사도: 0.6666666666666667
문서 2과 문서 3의 코사인 유사도: 1.0000000000000002


## 유사도를 이용한 추천 시스템 구현하기

영화를 입력하면 해당 영화의 줄거리와 유사한 줄거리의 영화를 찾아서 추천하는 시스템

In [2]:
import kagglehub

path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

Using Colab cache for faster access to the 'the-movies-dataset' dataset.


In [3]:
!ls {path}

credits.csv   links.csv        movies_metadata.csv  ratings_small.csv
keywords.csv  links_small.csv  ratings.csv


In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = pd.read_csv(path + "/movies_metadata.csv", low_memory=False)
data.head(2)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [5]:
data = data.head(20000)

In [6]:
print('overview 열의 결측값 개수:', data['overview'].isnull().sum())

overview 열의 결측값 개수: 135


In [7]:
data['overview'] = data['overview'].fillna('')

In [8]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['overview'])
print('TF-IDF 행렬의 크기:', tfidf_matrix.shape)
#20000개의 영화와 47847개의 단어가 사용됨

TF-IDF 행렬의 크기: (20000, 47487)


In [9]:
title_to_index = dict(zip(data['title'], data.index))
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [10]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = title_to_index[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return data['title'].iloc[movie_indices]

In [11]:
get_recommendations('The Dark Knight Rises')

,title
12481,The Dark Knight
150,Batman Forever
1328,Batman Returns
15511,Batman: Under the Red Hood
585,Batman
9230,Batman Beyond: Return of the Joker
18035,Batman: Year One
19792,"Batman: The Dark Knight Returns, Part 1"
3095,Batman: Mask of the Phantasm
10122,Batman Begins


In [12]:
get_recommendations('Toy Story')

,title
15348,Toy Story 3
2997,Toy Story 2
10301,The 40 Year Old Virgin
8327,The Champ
1071,Rebel Without a Cause
11399,For Your Consideration
1932,Condorman
3057,Man on the Moon
485,Malice
11606,Factory Girl


In [13]:
get_recommendations('Star Wars')

,title
1154,The Empire Strikes Back
1167,Return of the Jedi
1267,Mad Dog Time
5187,The Triumph of Love
309,The Swan Princess
461,Hot Shots! Part Deux
19400,Deathstalker II
3502,Shanghai Noon
13735,The Flame of New Orleans
1987,Sleeping Beauty
